# SIMULASI EXTRACT (3 BATCH PERIODIK)

In [1]:
import pandas as pd
import atoti as tt
import time

def extract_batch(batch_id):
    print(f"Extract batch {batch_id}")

    fact = pd.read_csv("fact_flight.csv")

    # simulasi filter data per batch (misalnya berdasarkan index/time)
    fact_batch = fact.sample(frac=0.7, random_state=batch_id)

    dim_date = pd.read_csv("dim_date.csv")
    dim_airline = pd.read_csv("dim_airline.csv")
    dim_origin = pd.read_csv("dim_origin.csv")
    dim_destination = pd.read_csv("dim_destination.csv")
    dim_delay = pd.read_csv("dim_delay.csv")
    dim_cancellation = pd.read_csv("dim_cancellation.csv")

    return fact_batch, dim_date, dim_airline, dim_origin, dim_destination, dim_delay, dim_cancellation

Welcome to Atoti 0.9.15!

By using this community edition, you agree with the license available at https://docs.activeviam.com/products/atoti/python-sdk/latest/eula.html.
Browse the official documentation at https://docs.activeviam.com/products/atoti/python-sdk.
Join the community at https://www.atoti.io/register.

Atoti collects telemetry data, which is used to help understand how to improve the product.
If you don't wish to send usage data, you can request a trial license at https://www.atoti.io/evaluation-license-request.

You can hide this message by setting the `ATOTI_HIDE_EULA_MESSAGE` environment variable to True.


# LOOP BATCH (SIMULASI PERIODIK)

In [2]:
all_batches = []

for i in range(1, 4):
    fact_batch, dim_date, dim_airline, dim_origin, dim_destination, dim_delay, dim_cancellation = extract_batch(i)
    fact_batch["BATCH_ID"] = i
    all_batches.append(fact_batch)

    time.sleep(1)

fact = pd.concat(all_batches, ignore_index=True)

Extract batch 1
Extract batch 2
Extract batch 3


# TRANSFORM

In [3]:
dim_date["YEAR"] = dim_date["YEAR"].astype(str)
dim_date["QUARTER"] = dim_date["QUARTER"].astype(str)
dim_date["MONTH"] = dim_date["MONTH"].astype(str)
dim_date["DAY"] = dim_date["DAY"].astype(str)

# LOAD KE ATOTI (DATA MART)

In [4]:
session = tt.Session.start()
print(session.url)

http://localhost:1918


In [5]:
fact_table = session.read_pandas(
    fact,
    table_name="fact_flight",
    keys=["FLIGHT_FACT_ID"]
)

date_table = session.read_pandas(
    dim_date,
    table_name="dim_date",
    keys=["DATE_ID"]
)

airline_table = session.read_pandas(
    dim_airline,
    table_name="dim_airline",
    keys=["AIRLINE_ID"]
)

origin_table = session.read_pandas(
    dim_origin,
    table_name="dim_origin",
    keys=["ORIGIN_ID"]
)

destination_table = session.read_pandas(
    dim_destination,
    table_name="dim_destination",
    keys=["DESTINATION_ID"]
)

delay_table = session.read_pandas(
    dim_delay,
    table_name="dim_delay",
    keys=["DELAY_ID"]
)

cancellation_table = session.read_pandas(
    dim_cancellation,
    table_name="dim_cancellation",
    keys=["CANCELLATION_ID"]
)

# RELASI STAR SCHEMA

In [6]:
fact_table.join(date_table)
fact_table.join(airline_table)
fact_table.join(origin_table)
fact_table.join(destination_table)
fact_table.join(delay_table)
fact_table.join(cancellation_table)

# CREATE CUBE

In [7]:
cube = session.create_cube(fact_table)

m = cube.measures
h = cube.hierarchies

# MEASURES (OLAP METRICS)

In [9]:
m["Avg Departure Delay"] = tt.agg.mean(fact_table["DEP_DELAY"])
m["Avg Arrival Delay"] = tt.agg.mean(fact_table["ARR_DELAY"])
m["Avg Air Time"] = tt.agg.mean(fact_table["AIR_TIME"])
m["Avg Distance"] = tt.agg.mean(fact_table["DISTANCE"])
m["Total Cancellation"] = tt.agg.sum(fact_table["CANCELLED"])
m["Total Flights"] = tt.agg.count_distinct(fact_table["FLIGHT_FACT_ID"])

# HIERARCHY (DIMENSION MODEL)

In [10]:
h["Date"] = [
    date_table["YEAR"],
    date_table["QUARTER"],
    date_table["MONTH_NAME"],
    date_table["DAY"]
]

h["Airline"] = [
    airline_table["AIRLINE"],
    airline_table["AIRLINE_CODE"]
]

h["Origin"] = [
    origin_table["ORIGIN_CITY"],
    origin_table["ORIGIN"]
]

h["Destination"] = [
    destination_table["DEST_CITY"],
    destination_table["DEST"]
]

h["Delay"] = [
    delay_table["DELAY_CATEGORY"]
]

h["Cancellation"] = [
    cancellation_table["CANCELLATION_REASON"]
]

# OLAP QUERY

In [18]:
# Rata-rata delay per airline
cube.query(
    m["Avg Departure Delay"],
    levels=[h["Airline"]["AIRLINE"]]
)

,Avg Departure Delay
AIRLINE,
Alaska Airlines Inc.,4.59
Allegiant Air,13.29
American Airlines Inc.,12.26
Delta Air Lines Inc.,8.01
Endeavor Air Inc.,5.83
Envoy Air,6.53
ExpressJet Airlines LLC d/b/a aha!,12.11
Frontier Airlines Inc.,15.65
Hawaiian Airlines Inc.,5.10


In [24]:
# Trend delay per bulan
cube.query(
    m["Avg Arrival Delay"],
    levels=[
        h["Date"]["YEAR"],
        h["Date"]["MONTH_NAME"]
    ]
)

Avg Arrival Delay
YEAR QUARTER MONTH_NAME                  
2019 1       February                8.79
             January                 4.02
             March                   3.25
     2       April                   4.43
             June                   11.55
             May                     6.39
     3       August                  7.55
             July                    8.09
             September                .08
     4       December                5.97
             November                -.12
             October                 2.34
2020 1       February                -.31
             January                -1.65
             March                  -5.23
     2       April                  -8.09
             June                   -7.61
             May                   -10.54
     3       August                 -5.97
             July                   -5.23
             September              -7.28
     4       December               -1.93
             November               -6.62
             October                -4.98
2021 1       February                 .42
             January                -4.95
             March                  -4.62
     2       April                  -3.70
             June                   10.20
             May                     -.53
     3       August                  8.78
             July                   11.64
             September                .36
     4       December                6.56
             November                 .65
             October                 3.84
2022 1       February                4.39
             January                 3.71
             March                   7.20
     2       April                   8.28
             June                   10.42
             May                     7.06
     3       August                  8.18
             July                    9.63
             September               1.88
     4       December               12.72
             November                4.49
             October                 2.02
2023 1       February                3.89
             January                 7.55
             March                   9.53
     2       April                   9.05
             June                   13.91
             May                     3.79
     3       August                  8.34
             July                   16.48

In [21]:
# Cancellation per airline
cube.query(
    m["Total Cancellation"],
    levels=[h["Airline"]["AIRLINE"]]
)

,Total Cancellation
AIRLINE,
Alaska Airlines Inc.,"1,875.00"
Allegiant Air,"2,394.00"
American Airlines Inc.,"10,606.00"
Delta Air Lines Inc.,"5,805.00"
Endeavor Air Inc.,"2,336.00"
Envoy Air,"3,543.00"
ExpressJet Airlines LLC d/b/a aha!,"1,041.00"
Frontier Airlines Inc.,"1,617.00"
Hawaiian Airlines Inc.,377.00


In [25]:
# Per airline (dimension analysis)
cube.query(
    m["Avg Departure Delay"],
    levels=[h["Airline"]["AIRLINE"]]
)

,Avg Departure Delay
AIRLINE,
Alaska Airlines Inc.,4.59
Allegiant Air,13.29
American Airlines Inc.,12.26
Delta Air Lines Inc.,8.01
Endeavor Air Inc.,5.83
Envoy Air,6.53
ExpressJet Airlines LLC d/b/a aha!,12.11
Frontier Airlines Inc.,15.65
Hawaiian Airlines Inc.,5.10


In [26]:
# Trend waktu (time series)
cube.query(
    m["Avg Arrival Delay"],
    levels=[h["Date"]["YEAR"], h["Date"]["MONTH_NAME"]]
)

Avg Arrival Delay
YEAR QUARTER MONTH_NAME                  
2019 1       February                8.79
             January                 4.02
             March                   3.25
     2       April                   4.43
             June                   11.55
             May                     6.39
     3       August                  7.55
             July                    8.09
             September                .08
     4       December                5.97
             November                -.12
             October                 2.34
2020 1       February                -.31
             January                -1.65
             March                  -5.23
     2       April                  -8.09
             June                   -7.61
             May                   -10.54
     3       August                 -5.97
             July                   -5.23
             September              -7.28
     4       December               -1.93
             November               -6.62
             October                -4.98
2021 1       February                 .42
             January                -4.95
             March                  -4.62
     2       April                  -3.70
             June                   10.20
             May                     -.53
     3       August                  8.78
             July                   11.64
             September                .36
     4       December                6.56
             November                 .65
             October                 3.84
2022 1       February                4.39
             January                 3.71
             March                   7.20
     2       April                   8.28
             June                   10.42
             May                     7.06
     3       August                  8.18
             July                    9.63
             September               1.88
     4       December               12.72
             November                4.49
             October                 2.02
2023 1       February                3.89
             January                 7.55
             March                   9.53
     2       April                   9.05
             June                   13.91
             May                     3.79
     3       August                  8.34
             July                   16.48

In [27]:
# KPI bisnis (cancellation)
cube.query(
    m["Total Cancellation"],
    levels=[h["Airline"]["AIRLINE"]]
)

,Total Cancellation
AIRLINE,
Alaska Airlines Inc.,"1,875.00"
Allegiant Air,"2,394.00"
American Airlines Inc.,"10,606.00"
Delta Air Lines Inc.,"5,805.00"
Endeavor Air Inc.,"2,336.00"
Envoy Air,"3,543.00"
ExpressJet Airlines LLC d/b/a aha!,"1,041.00"
Frontier Airlines Inc.,"1,617.00"
Hawaiian Airlines Inc.,377.00


# Masuk Dashboard Atoti

In [28]:
session.url

'http://localhost:1918'

In [29]:
session.close()

Berdasarkan seluruh hasil analisis yang telah dilakukan, diperoleh beberapa insight penting.

Pertama, rata-rata keterlambatan keberangkatan lebih tinggi dibandingkan keterlambatan kedatangan.

Kedua, JetBlue Airways dan Allegiant Air merupakan maskapai dengan tingkat keterlambatan tertinggi.

Ketiga, American Airlines dan Southwest Airlines memiliki jumlah pembatalan penerbangan terbesar.

Keempat, faktor cuaca merupakan penyebab utama pembatalan penerbangan.

Kelima, tren keterlambatan menunjukkan peningkatan dari tahun 2021 hingga 2023 yang mengindikasikan meningkatnya kepadatan operasional penerbangan.